# Sensors Analytics REST API

## Deploy the API
- Go to the *Machine* tab, click the *More Options* icon, then set *Incoming connections* to **On**. The API will be accessible through the indicated tunnelling link.  
- Run the notebook.



## Import the required modules

In [8]:
import cherrypy
import json
import redis

## Connect to the Redis Database

In [9]:
# Configure here your Redis connection parameters
REDIS_HOST = "redis-15448.c300.eu-central-1-1.ec2.cloud.redislabs.com"
REDIS_PORT = "15448"
REDIS_USER = "default"
REDIS_PASS = "QVmVhSJy9xbmT9qtySmpMFCgWQKx0otn"

redis_client = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, username=REDIS_USER, password=REDIS_PASS)

is_connected = redis_client.ping()
print('Redis Connected:', is_connected)

Redis Connected: True


## Historical data resource

In [10]:
class HistoricalData(object):
    exposed = True
    
    def GET(self, *path, **query):
        # Validate MAC address
        if len(path) != 1:
            raise cherrypy.HTTPError(400, "Missing MAC address.")

        mac_address = path[0]

        # Validate count parameter
        if "count" not in query:
            raise cherrypy.HTTPError(400, "Missing count parameter.")

        try:
            count = int(query["count"])
        except ValueError:
            raise cherrypy.HTTPError(400, "Count must be an integer.")

        if count <= 0:
            raise cherrypy.HTTPError(400, "The value of count must be a positive integer.")

        humidity_ts = f"{mac_address}:humidity"
        temperature_ts = f"{mac_address}:temperature"

        # Check existence in Redis
        if not redis_client.exists(humidity_ts) or not redis_client.exists(temperature_ts):
            raise cherrypy.HTTPError(404, "MAC address not found in the database.")

        # Fetch latest samples
        humidities_data = redis_client.ts().revrange(
            key=humidity_ts,
            from_time="-",
            to_time="+",
            count=count
        )

        temperatures_data = redis_client.ts().revrange(
            key=temperature_ts,
            from_time="-",
            to_time="+",
            count=count
        )

        # Ensure both time series have data
        if not humidities_data or not temperatures_data:
            raise cherrypy.HTTPError(404, "No historical data available.")

        # Align lengths safely
        n = min(len(humidities_data), len(temperatures_data))

        timestamp = []
        temperature = []
        humidity = []

        for i in range(n):
            ts_temp, temp = temperatures_data[i]
            ts_hum, hum = humidities_data[i]

            # Use temperature timestamp as reference
            timestamp.append(ts_temp)
            temperature.append(temp)
            humidity.append(hum)

        response_dict = {
            "mac_address": mac_address,
            "timestamp": timestamp,
            "temperature": temperature,
            "humidity": humidity
        }
        
        return json.dumps(response_dict)
    

## Status Endpoint Class

In [11]:
class Status(object):
    exposed = True

    def GET(self, *path, **query):
        response_dict = {
            'status': 'online'
        }
        response = json.dumps(response_dict)

        return response

## Sensors Endpoint Class

In [12]:
class Sensors(object):
    exposed = True

    def GET(self, *path, **query):
        min_t_samples = int(query.get('min_t_samples', 0))
        min_h_samples = int(query.get('min_h_samples', 0))
        sensors = []
        keys = redis_client.keys('0x*:temperature')

        count = 0
        for key in keys:
            key = key.decode()
            mac_address = key.split(':')[0]

            t_info = redis_client.ts().info(f'{mac_address}:temperature')
            t_samples = t_info.total_samples
            t_retention = t_info.retention_msecs
            h_info = redis_client.ts().info(f'{mac_address}:humidity')
            h_samples = h_info.total_samples
            h_retention = h_info.retention_msecs

            if t_samples >= min_t_samples and h_samples >= min_h_samples:
                sensors.append(
                    {
                        "mac_address": mac_address,
                        "t_samples": t_samples,
                        "t_retention": t_retention,
                        "h_samples": h_samples,
                        "h_retention": h_retention,
                    }
                )
                count += 1

        response_dict = {
            "sensors": sensors,
            "count": count,
        }

        response = json.dumps(response_dict)

        return response

    def POST(self, *path, **query):
        body = cherrypy.request.body.read()
        body_dict = json.loads(body.decode())

        mac_address = body_dict.get('mac_address', None)

        if mac_address is None:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request body.')

        try:
            redis_client.ts().create(f'{mac_address}:temperature', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        try:
            redis_client.ts().create(f'{mac_address}:humidity', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        return

## Sensor Endpoint Class

In [13]:
class Sensor(object):
    exposed = True

    def GET(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        h_info = redis_client.ts().info(f'{mac_address}:humidity')

        response_dict = {
            "mac_address": mac_address,
            "t_samples": t_info.total_samples,
            "t_retention": t_info.retention_msecs,
            "h_samples": h_info.total_samples,
            "h_retention": h_info.retention_msecs,
        }

        response = json.dumps(response_dict)

        return response

    def PUT(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        body = cherrypy.request.body.read()
        body_dict = json.loads(body.decode())

        t_retention = body_dict.get('t_retention', None)

        if t_retention is None:
            raise cherrypy.HTTPError(400, 'Missing temperature retention period in the request body.')

        h_retention = body_dict.get('h_retention', None)

        if h_retention is None:
            raise cherrypy.HTTPError(400, 'Missing humidity retention period in the request body.')

        redis_client.ts().alter(f'{mac_address}:temperature', retention_msecs=t_retention)
        redis_client.ts().alter(f'{mac_address}:humidity', retention_msecs=h_retention)

        return

    def DELETE(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]
        found = 0
        found += redis_client.delete(f'{mac_address}:temperature')
        found += redis_client.delete(f'{mac_address}:humidity')

        if found == 0:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        return

## Setup cherrypy and Map objects to their target endpoints

In [ ]:
if __name__ == '__main__':
    conf = {'/': {'request.dispatch': cherrypy.dispatch.MethodDispatcher()}}
    cherrypy.tree.mount(Status(), '/status', conf)
    cherrypy.tree.mount(Sensors(), '/sensors', conf)
    cherrypy.tree.mount(Sensor(), '/sensor', conf)
    cherrypy.tree.mount(HistoricalData(), '/data', conf)
    cherrypy.config.update({'server.socket_host': '0.0.0.0'})
    cherrypy.config.update({'server.socket_port': 8080})
    cherrypy.engine.start()
    cherrypy.engine.block()

[30/Jan/2026:12:40:45] ENGINE Bus STARTING
[30/Jan/2026:12:40:45] ENGINE Started monitor thread 'Autoreloader'.
[30/Jan/2026:12:40:45] ENGINE Serving on http://0.0.0.0:8080
[30/Jan/2026:12:40:45] ENGINE Bus STARTED


127.0.0.1 - - [30/Jan/2026:12:40:55] "GET /status HTTP/1.1" 200 20 "" "python-requests/2.32.5"
127.0.0.1 - - [30/Jan/2026:12:40:55] "POST /sensors HTTP/1.1" 409 3230 "" "python-requests/2.32.5"
127.0.0.1 - - [30/Jan/2026:12:40:55] "GET /data/0xe45f01d8f381?count=10 HTTP/1.1" 200 349 "" "python-requests/2.32.5"


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3880e510-b64c-4bb5-b488-c2122d5d9e2d' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>